# Data Pipelines for Pre-Training Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Text Cleaning

Strip HTML, normalize whitespace, remove non-text content. We will use a public domain text (Project Gutenberg) as our small corpus.

In [ ]:
```python

import re

def clean_text(text):

    text = re.sub(r"<[^>]+>", "", text)

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"[^\x20-\x7E\n]", "", text)

    text = re.sub(r"\n{3,}", "\n\n", text)

    text = re.sub(r" {2,}", " ", text)

    return text.strip()

def quality_filter(text, min_words=50, max_ratio_caps=0.3, max_ratio_special=0.1):

    words = text.split()

    if len(words) < min_words:

        return False

    caps_ratio = sum(1 for w in words if w.isupper()) / len(words)

    if caps_ratio > max_ratio_caps:

        return False

    special_chars = sum(1 for c in text if not c.isalnum() and not c.isspace())

    if special_chars / max(len(text), 1) > max_ratio_special:

        return False

    return True

In [ ]:
```

The quality filter catches SEO spam (ALL CAPS), machine-generated noise (high special character ratio), and stub pages (too short). These three checks alone remove a surprising amount of garbage from web crawls.

### Step 2: MinHash Deduplication

Implement MinHash from scratch. No external libraries required -- just `hashlib`.

In [ ]:
```python

import hashlib

from collections import defaultdict

def get_shingles(text, k=5):

    words = text.lower().split()

    if len(words) < k:

        return set()

    return {" ".join(words[i:i+k]) for i in range(len(words) - k + 1)}

def minhash_signature(shingles, num_hashes=128):

    signature = []

    for i in range(num_hashes):

        min_hash = float("inf")

        for shingle in shingles:

            h = int(hashlib.sha256(f"{i}:{shingle}".encode()).hexdigest(), 16)

            min_hash = min(min_hash, h)

        signature.append(min_hash)

    return signature

def lsh_buckets(signature, bands=16):

    rows_per_band = len(signature) // bands

    buckets = []

    for b in range(bands):

        start = b * rows_per_band

        band_data = tuple(signature[start:start + rows_per_band])

        bucket_hash = hashlib.md5(str(band_data).encode()).hexdigest()

        buckets.append((b, bucket_hash))

    return buckets

def deduplicate(documents, threshold=0.8, num_hashes=128, bands=16):

    signatures = []

    shingle_sets = []

    for doc in documents:

        shingles = get_shingles(doc)

        shingle_sets.append(shingles)

        signatures.append(minhash_signature(shingles, num_hashes))

    bucket_map = defaultdict(list)

    for doc_idx, sig in enumerate(signatures):

        for band_id, bucket_hash in lsh_buckets(sig, bands):

            bucket_map[(band_id, bucket_hash)].append(doc_idx)

    duplicate_pairs = set()

    for bucket_docs in bucket_map.values():

        if len(bucket_docs) < 2:

            continue

        for i in range(len(bucket_docs)):

            for j in range(i + 1, len(bucket_docs)):

                duplicate_pairs.add((bucket_docs[i], bucket_docs[j]))

    removed = set()

    for i, j in duplicate_pairs:

        if i in removed or j in removed:

            continue

        s1, s2 = shingle_sets[i], shingle_sets[j]

        if not s1 or not s2:

            continue

        jaccard = len(s1 & s2) / len(s1 | s2)

        if jaccard >= threshold:

            removed.add(j)

    return [doc for idx, doc in enumerate(documents) if idx not in removed], len(removed)

In [ ]:
```

The `num_hashes=128` and `bands=16` parameters control the precision-recall tradeoff. More hashes give more accurate similarity estimates. More bands increase recall (catch more duplicates) at the cost of more false positives. These values work well for typical web text.

### Step 3: Tokenize and Pack Sequences

Take the clean, deduplicated text, tokenize it, and pack into fixed-length sequences for training.

In [ ]:
```python

def tokenize_corpus(documents, tokenizer):

    all_tokens = []

    for doc in documents:

        tokens = tokenizer.encode(doc)

        all_tokens.extend(tokens)

        all_tokens.append(tokenizer.eos_id)

    return all_tokens

def pack_sequences(token_ids, seq_length, pad_id=0):

    sequences = []

    attention_masks = []

    for i in range(0, len(token_ids), seq_length):

        seq = token_ids[i:i + seq_length]

        mask = [1] * len(seq)

        if len(seq) < seq_length:

            pad_count = seq_length - len(seq)

            seq = seq + [pad_id] * pad_count

            mask = mask + [0] * pad_count

        sequences.append(seq)

        attention_masks.append(mask)

    return sequences, attention_masks

In [ ]:
```

### Step 4: DataLoader for Training

Yield randomized batches of packed sequences. This is what the training loop consumes.

In [ ]:
```python

import random

class PreTrainingDataLoader:

    def __init__(self, sequences, attention_masks, batch_size, shuffle=True):

        self.sequences = sequences

        self.attention_masks = attention_masks

        self.batch_size = batch_size

        self.shuffle = shuffle

    def __len__(self):

        return (len(self.sequences) + self.batch_size - 1) // self.batch_size

    def __iter__(self):

        indices = list(range(len(self.sequences)))

        if self.shuffle:

            random.shuffle(indices)

        for start in range(0, len(indices), self.batch_size):

            batch_idx = indices[start:start + self.batch_size]

            batch_seqs = [self.sequences[i] for i in batch_idx]

            batch_masks = [self.attention_masks[i] for i in batch_idx]

            yield batch_seqs, batch_masks

In [ ]:
```

### Step 5: Dataset Statistics

Compute the numbers that matter: total tokens, unique tokens, compression ratio, document length distribution.

In [ ]:
```python

from collections import Counter

def compute_statistics(documents, token_ids, sequences, tokenizer_vocab_size):

    total_chars = sum(len(d) for d in documents)

    total_tokens = len(token_ids)

    unique_tokens = len(set(token_ids))

    compression_ratio = total_chars / total_tokens

    doc_lengths = [len(d.split()) for d in documents]

    avg_doc_length = sum(doc_lengths) / max(len(doc_lengths), 1)

    max_doc_length = max(doc_lengths) if doc_lengths else 0

    min_doc_length = min(doc_lengths) if doc_lengths else 0

    token_counts = Counter(token_ids)

    top_tokens = token_counts.most_common(10)

    non_pad_tokens = sum(sum(1 for t in seq if t != 0) for seq in sequences)

    total_positions = sum(len(seq) for seq in sequences)

    utilization = non_pad_tokens / max(total_positions, 1)

    stats = {

        "total_documents": len(documents),

        "total_characters": total_chars,

        "total_tokens": total_tokens,

        "unique_tokens": unique_tokens,

        "vocab_utilization": unique_tokens / tokenizer_vocab_size,

        "compression_ratio": compression_ratio,

        "avg_doc_length_words": avg_doc_length,

        "max_doc_length_words": max_doc_length,

        "min_doc_length_words": min_doc_length,

        "num_sequences": len(sequences),

        "sequence_utilization": utilization,

        "top_10_tokens": top_tokens,

    }

    return stats

In [ ]:
```

Compression ratio tells you how efficient the tokenizer is on this corpus. English text typically compresses to about 3-4 characters per token. If you see 1.5 characters per token, your tokenizer is splitting too aggressively. If you see 8+, it has learned very domain-specific merges.

Sequence utilization tells you how much of your packed sequences is real data versus padding. Below 90% means your packing is inefficient -- you are wasting compute on padding tokens.

## Exercises

In [ ]:
1. **Easy:** Add language detection to the cleaning pipeline using a simple heuristic (character set analysis). Filter to only English documents and measure how many documents get removed.
2. **Medium:** Implement exact deduplication using SHA-256 hashes alongside the MinHash near-deduplication. Compare the number of duplicates caught by each method on a web-scraped corpus.
3. **Hard:** Build a perplexity-based quality filter. Train a small bigram language model on Wikipedia text, score each document by perplexity, and remove the bottom 20%. Compare model output quality when training on filtered vs unfiltered data.